# အလယ်အလတ် အဖွဲ့ဝင် Middleware ဖြင့် ဟိုတယ် ဘွတ်ကင်လုပ်ခြင်း

ဒီ notebook က Microsoft Agent Framework ကို အသုံးပြုပြီး **function-based middleware** ကို ပြသပါတယ်။ ကျွန်တော်တို့ conditional workflow နမူနာပေါ်မှာ အခြေခံပြီး အလယ်အလတ် အဖွဲ့ဝင်တွေကို အထူးအခွင့်အရေးပေးတဲ့ middleware အလွှာတစ်ခု ထည့်သွင်းထားပါတယ်။

## သင်လေ့လာရမည့်အချက်များ:
1. **Function-Based Middleware**: function ရလဒ်များကို အတက်အကျ ချိုးမောင်းခြင်းနှင့် ပြင်ဆင်ခြင်း
2. **Context Access**: အလုပ်လုပ်ပြီးပြီးနောက် `context.result` ကို ဖတ်ရှုခြင်းနှင့် ပြင်ဆင်ခြင်း
3. **စီးပွားရေး လုပ်ငန်းထဲက Logic အကောင်အထည်ဖော်ခြင်း**: အလယ်အလတ် အဖွဲ့ဝင်အကျိုးခံစားခွင့်များ
4. **ရလဒ်ပြောင်းလဲမှု**: အသုံးပြုသူအခြေအနေအပေါ်မူတည်၍ function ရလဒ်ပေါ်ပြောင်းလဲခြင်း
5. **တူညီ Workflow, ကွဲပြားသောရလဒ်များ**: Middleware အားဖြင့် မူလီအပြောင်းအလဲများ

## Middleware ထည့်သွင်းထားသော Workflow အင်ဂျင်:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## Conditional Workflow နှင့် ကွဲပြားချက် အဓိက:

**Middleware မပါသောအခါ** (14-conditional-workflow.ipynb):
- ပဲရစ်မှာ အခန်းမရှိ → alternative_agent ကို ပို့ရန်

**Middleware ပါသောအခါ** (ဒီ notebook):
- ပုံမှန် အသုံးပြုသူ + ပဲရစ် → အခန်း မရှိ → alternative_agent ကို ပို့ရန်
- အလယ်အလတ် အသုံးပြုသူ + ပဲရစ် → 🌟 Middleware override ဖြစ်သည်! → အခန်း ရရှိနိုင် → booking_agent ကို ပို့ရန်

## ကြိုတင်လိုအပ်ချက်များ:
- Microsoft Agent Framework တပ်ဆင်ပြီး
- conditional workflows ကို နားလည်မှု (14-conditional-workflow.ipynb ကို ကြည့်ပါ)
- GitHub token သို့မဟုတ် OpenAI API key
- middleware ပုံစံများအပေါ် အခြေခံ နားလည်မှု


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## အဆင့် ၁: ဖွဲ့စည်းထားသော ထုတ်လွှင့်ချက်များအတွက် Pydantic မော်ဒယ်များကို သတ်မှတ်ပါ

ဤမော်ဒယ်များသည် အေးဂျင့်များ ပြန်လည်ပေးအပ်မည့် **schema** ကို သတ်မှတ်သည်။ ပြင်ဆင်မှု မီဒီယာသည် ရရှိနိုင်မှု ရလဒ်ကို ပြင်ဆင်သည့်အခါ ကို စောင့်ကြည့်ရန် `priority_override` လက်နက်ဧရိယာကို ထည့်သွင်းထားသည်။


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## အဆင့် ၂: ဦးစားပေး အဖွဲ့ဝင်များ ဒေတာဘဲစ်ကို သတ်မှတ်ခြင်း

ဒီ ဒမ်းမိုအတွက် ကျွန်တော်တို့က ဦးစားပေး အဖွဲ့ဝင် ဒေတာဘဲစ်ကို သရုပ်ဆောင် ပြသမှာ ဖြစ်ပါတယ်။ ထုတ်လုပ်မှုပတ်ဝန်းကျင်တွင် အမှန်တကယ် ဒေတာဘဲစ် သို့မဟုတ် API ကို စစ်ဆေးမှာ ဖြစ်ပါတယ်။

**ဦးစားပေး အဖွဲ့ဝင်များ:**
- `alice@example.com` - VIP အဖွဲ့ဝင်
- `bob@example.com` - Premium အဖွဲ့ဝင်  
- `priority_user` - စမ်းသပ် အကောင့်


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## အဆင့် ၃: ဟိုတယ်မှာကြိုတင်မှာယူတဲ့သုံးစွဲကိရိယာကို ဖန်တီးပါ

အခြေအနေပါဝင်သော workflow နဲ့ တူတူပဲ ဖြစ်ပေမယ့် ယခုမှာ middleware မှတဆင့် ဖမ်းဆီးခံရမည်ဖြစ်သည်!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## အဆင့်(၄): 🌟 ဦးစားပေးစစ်ဆေးမှု Middleware ဖန်တီးပါ (အဓိကအချက်!)

ဒီ notebook ရဲ့ **အဓိကလုပ်ဆောင်မှု** ဖြစ်သည်။ Middleware သည်–

1. hotel_booking function ခေါ်ဆိုမှုကို **ဖမ်းဆီးသည်**
2. `next(context)` ကို ခေါ်ဆိုပြီး function ကို ပုံမှန်အတိုင်း **အကောင်အထည်ဖော်သည်**
3. `context.result` ထဲရှိ အကျိုးသက်ရောက်မှုကို **စစ်ဆေးသည်**
4. အသုံးပြုသူသည် ဦးစားပေးဖြစ်ပြီး အခန်း မရှိသောအခါ အကျိုးသက်ရောက်မှုကို **အလျော့ပြင်ဆင်သည်**
5. ပြုပြင်ပြောင်းလဲထားသော အကျိုးသက်ရောက်မှုကို agent ထံ **ပြန်လည်ပို့ပေးသည်**

**အဓိကပုံစံ:**
```python
async def my_middleware(context, next):
    await next(context)  # ဖင်ရှင်ကို အကောင်အထည်ဖော်ပါ
    # ယခု context.result တွင် ဖင်ရှင်ရလဒ်ကိုပါဝင်သည်
    if some_condition:
        context.result = new_value  # အစားထိုးပါ!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## အဆင့် ၅: လမ်းကြောင်းသတ်မှတ်ရန် အနေအထားဖန်ဆင်းမှုများကို သတ်မှတ်ပါ

အခြေအနေဖန်ဆင်းမှုများသည် အခြေအနေဆိုင်ရာ workflow နှင့် တူညီသည် - ၎င်းတို့သည် လမ်းကြောင်းသတ်မှတ်ရန် ဖွဲ့စည်းထားသော ထွက်ရှိမှုကို စစ်ဆေးသည်။


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## အဆင့် ၆: စိတ်ကြိုက် ပြသခြင်း အကောင်အထည်ဖွဲ့သူ ဖန်တီးခြင်း

အရင်တုန်းကအသုံးပြုခဲ့တဲ့အကောင်အထည်ဖွဲ့သူတူညီပါသည် - အလုပ်စဥ် အဆုံးသတ် output ကိုပြသသည်။


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## အဆင့် ၇: ပတ်ဝန်းကျင် အသက်သွင်းကြောင်းများကို ဆောင်ရွက်ပါ

LLM client (Microsoft Foundry / Azure OpenAI) ကို စီစဉ်ထားပါ။


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## အဆင့် ၈: Middleware ဖြင့် AI အေးဂျင့်များ ဖန်တီးခြင်း

**အဓိက ကွာခြားချက်:** availability_agent ကို ဖန်တီးသည့်အခါ `middleware` ပါရာမီတာကို ပေးပို့သည်!

၎င်းသည် priority_check_middleware ကို အေးဂျင့်၏ function invocation pipeline ထဲသို့ ထည့်သွင်းပေးသည့် နည်းလမ်းဖြစ်သည်။


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## အဆင့် ၉: Workflow ကို ဆောက်ရန်

ယခင်ကဲ့သို့ workflow ဖွဲ့စည်းမှုတူညီသည် - ရရှိနိုင်မှုအပေါ် အခြေခံ၍ အခြေအနေလိုက် လမ်းကြောင်း သတ်မှတ်ခြင်း။


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## အဆင့် ၁၀: စမ်းသပ်မှု ကိစ္စ ၁ - ပဲရစ်ရှိ ပုံမှန် အသုံးပြုသူ (မပိုလည်းမချည်း)

ပုံမှန် အသုံးပြုသူတစ်ဦးက ပဲရစ်ဘွတ်ခ်ကြည့်သည် → အခန်း မရှိ → alternative_agent သို့ လမ်းကြောင်းပြောင်းသည်


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## အဆင့် ၁၁: စမ်းသပ်မှုအမှုစွဲ ၂ - 🌟 ဦးစားပေး အသုံးပြုသူ in ပဲရစ် (Override ပါရှိသည်!)

ဦးစားပေး အဖွဲ့ဝင်တစ်ဦးသည် ပဲရစ်ကို စာရင်းတင်ရန် ကြိုးပမ်းသည် → မူလတွင် အခန်းများ မရှိ → 🌟 Middleware က override လုပ်သည်! → booking_agent သို့ လမ်းညွှန်သည်

**ဒါဟာ middleware အင်အား၏ အဓိက ပြသချက် ဖြစ်သည်!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## အဆင့် ၁၂: စမ်းသပ်မှုကိစ္စ ၃ - စတောက်ဟိုးလမ်ရှိ ဦးစားပေးအသုံးပြုသူ (ပြီးသား ရရှိနိုင်)

ဦးစားပေးအသုံးပြုသူသည် စတောက်ဟိုးလမ်ကို ကြိုးစားသည် → အခန်းများ ရရှိနိုင်သည် → အစားထိုးရန် မလိုအပ်သည် → booking_agent ဆီသို့ လမ်းကြောင်းညွှန်ပြသည်

၎င်းသည် မီဒီအာဝဲ မလိုအပ်သောအခါတွင်သာ လုပ်ဆောင်ကြောင်း ပြသပါသည်!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## အဓိက အစိတ်အပိုင်းများနှင့် Middleware အယူအဆများ

### ✅ သင် သင်ခွင့်ရသည်

#### **၁။ ဖန်ခွဲမှုအခြေခံ Middleware ပုံစံ**

Middleware သည် လွယ်ကူသော async function ကို အသုံးပြု၍ function ခေါ်ဆိုမှုများကို ဖမ်းဆီးသည်။

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # ဂဏန်းလုပ်ဆောင်မှုမတိုင်မှီ
    print("Intercepting...")
    
    # ဂဏန်းကိုဆောင်ရွက်ရန်
    await next(context)
    
    # ဂဏန်းလုပ်ဆောင်ပြီးနောက် - ရလဒ်ကိုစစ်ဆေးပါ
    if context.result:
        # လိုအပ်ပါက ရလဒ်ကိုပြင်ဆင်ပါ
        context.result = modified_value
```

#### **၂။ Context ဝင်ရောက်ခြင်းနှင့် ရလဒ် ပြုပြင်ခြင်း**

- `context.function` - ခေါ်ဆိုနေသော function ကို 접근ရန်
- `context.arguments` - function ၏ argument များကို ဖတ်ရန်
- `context.kwargs` - ပို၍ parameters များကို 접근ရန်
- `await next(context)` - function ကို အလုပ်လုပ်စေမည်
- `context.result` - function ၏ output ကို ဖတ်/ပြုပြင်ရန်

#### **၃။ စီးပွားရေး Logic ကို အကောင်အထည်ဖော်ခြင်း**

ကျွန်ုပ်တို့၏ middleware သည် ဦးစားပေး အဖွဲ့ဝင် အကျိူးခံစားခွင့်များကို ထည့်သွင်းဆောင်ရွက်သည်။
- **ပုံမှန် အသုံးပြုသူများ**: ပြင်ဆင်မှုမရှိ သာမာန် လုပ်ငန်းစဉ်များ
- **ဦးစားပေး အသုံးပြုသူများ**: "မရရှိနိုင်ပါ" → "ရရှိနိုင်ပါပြီ" အစားထိုးသည်
- **သတ်မှတ်ချက် Logic**: လိုအပ်သမျှအချိန်တွင်သာ အသုံးပြုသည်

#### **၄။ တူညီသော လုပ်ငန်းစဉ်၊ ကွဲပြားသော ရလဒ်များ**

Middleware ၏ အင်အား:
- ✅ လုပ်ငန်းစဉ် ဖွဲ့စည်းမှု တည်းမပြောင်းပါ
- ✅ ကိရိယာ function တည်းမပြောင်းပါ
- ✅ သတ်မှတ်ချက် အခြေအနေ routing logic တည်းမပြောင်းပါ
- ✅ Middleware သာ → ကွဲပြားသော အပြုအမူ!

### 🚀လက်တွေ့ အသုံးချမှုများ

၁။ **VIP/Premium အသုံးပြုသူအင်္ဂါရပ်များ**
   - Premium အသုံးပြုသူများအတွက် rate limit များကို ပြင်ဆင်ခြင်း
   - အရင်းအမြစ်များကို ဦးစားပေးဝင်ရောက်ခွင့်ပေးခြင်း
   - Premium အင်္ဂါရပ်များကို တိုးချဲ့လိုသည်အတိုင်း ဖြေရှင်းပေးခြင်း

၂။ **A/B စမ်းသပ်ခြင်း**
   - အသုံးပြုသူများကို ကွဲပြားသော အကောင်အထည်ဖော်မှုများ သို့ လမ်းညွှန်ခြင်း
   - အသေးစိတ်အသုံးပြုသူများနှင့် အသစ်များကို စမ်းသပ်ခြင်း
   - အဆင့်ဆင့် အသစ်များ ထည့်သွင်းခြင်း

၃။ **လုံခြုံမှုနှင့် ဝန်လိုက်မှု သတ်မှတ်ချက်များ**
   - function ခေါ်ဆိုမှုများကို စစ်ဆေးခြင်း
   - အရေးကြီး လုပ်ဆောင်ချက်များကို ပိတ်ပင်ဆောင်ရွက်ခြင်း
   - စီးပွားရေး စည်းမျဥ်းများ ထိန်းသိမ်းခြင်း

၄။ **စွမ်းဆောင်ရည် တိုးတက်မှု**
   - အသုံးပြုသူ အတိအကျများအတွက် ရလဒ်များကို ကုဒ်ပတ်ထဲသို့ သိမ်းဆည်းခြင်း
   - သိပ်သည်းသော လုပ်ဆောင်မှုများကို ဆန့်အုပ်နိုင်လျှင် လျှော့ချခြင်း
   - အရင်းအမြစ်များကို အလိုအလျောက် ချိန်ညှိပေးခြင်း

၅။ **အမှားကိုင်တွယ်မှုနှင့် ထပ်မံကြိုးစားမှု**
   - အမှားများကို သိမ်းပိုက်နှင့် ကိုင်တွယ်ပေးခြင်း
   - ထပ်မံကြိုးစားရေး ကိုလုပ်ေဆာင္သည့် Logic ကို လုပ်ထုံးလုပ်နည်းထိန်းသိမ်းခြင်း
   - အခြား မဟုတ်မဖြစ် အကောင်အထည်ဖော်မှုများသို့ ပြောင်းရွှေ့ခြင်း

၆။ **မှတ်တမ်းတင်ခြင်းနှင့် ကြည့်ရှုမှု**
   - function လုပ်ဆောင်မှု အချိန်များကို မှတ်တမ်းတင်ခြင်း
   - parameters များနှင့် ရလဒ်များ ကို မှတ်တမ်းတင်ခြင်း
   - အသုံးပြုမှု ပုံစံများကို စောင့်ကြည့်ခြင်း

### 🔑 Decorator မတူညီသော အချက်များ

| အင်္ဂါရပ် | Decorator | Middleware |
|---------|-----------|------------|
| **Scope** | တစ်ခုတည်းသော function | agent တွင်ရှိသည့် function အားလုံး |
| **အားသာချက်** | သတ်မှတ်ချိန်တွင် မပြောင်းလဲနိုင် | အလုပ်လုပ်ချိန်တွင် ပြောင်းလဲနိုင် |
| **Context** | အကန့်အသတ်ရှိ | အပြည့်အစုံ agent context |
| **ဖွဲ့စည်းမှု** | decorator များစွာ | Middleware Pipeline |
| **Agent အသိ** | မရှိ | ရှိ (agent အခြေအနေကို 접근နိုင်) |

### 📚 Middleware ကို ဘယ်အချိန် အသုံးပြုသင့်သလဲ

✅ **Middleware ကို အသုံးပြုပါက:**
- အသုံးပြုသူ/တိုက်ဆိုင်မှု အခြေအနေအရ အပြုအမှုကို ပြောင်းလဲရန်လိုသောအခါ
- function များစွာတွင် Logic ကို အသုံးပြုလိုသောအခါ
- agent အဆင့် Context ကို လိုအပ်သောအခါ
- တွဲဖက်ထည့်သွင်းမှုများ (logging, ခွင့်ပြုချက်စစ်ဆေးမှု စသည့်) အကောင်အထည်ဖော်သည့်အခါ

❌ **Middleware ကို မအသုံးပြုသင့်သော အခါ:**
- လွယ်ကူသော input မှန်ကန်မှုစစ်စာ (Pydantic ကို အသုံးပြုရန်)
- function ထဲတွင်သာ ထိန်းသိမ်းသင့်သော logic များ
- တစ်ခုတည်းပြောင်းလဲမှုများ (function ကိုသာ ပြောင်းရန်)

### 🎓 အဆင့်မြင့် ပုံစံများ

```python
# မျိုးစုံလက်တွေ့လုပ်ဆောင်မှုအလယ်ခန်းများ (အလည်အပြန် ရေးရာအရေးကြီးသည်)
middleware=[
    logging_middleware,      # စတင်မှာ မှတ်တမ်းတင်သည်
    auth_middleware,         # ထို့နောက် authentication ကို စစ်ဆေးသည်
    cache_middleware,        # ထို့နောက် cache ကို စစ်ဆေးသည်
    rate_limit_middleware,   # ထို့နောက် အမြန်နှုန်းကန့်သတ်သည်
    priority_check_middleware  # နောက်ဆုံးတွင် ဦးစားပေး စစ်ဆေးမှု
]

# အခြေအနေတစ်ခုအပေါ် မတူညီသော လက်တွေ့လုပ်ဆောင်မှုလုပ်ငန်းတော်
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # ရလဒ်ကို ပြောင်းလဲသည်
    else:
        # စုံစမ်းမှု လုံးဝကျော်လွှားသည်
        context.result = cached_value
```

### 🔗 ဆက်စပ်အယူအဆများ

- **Agent Middleware**: agent.run() ခေါ်ဆိုမှုများကို ဖမ်းဆီးသည်
- **Function Middleware**: ကိရိယာ function ခေါ်ဆိုမှုများကို ဖမ်းဆီးသည် (ကျွန်ုပ်တို့အသုံးပြုပုံ)
- **Middleware Pipeline**: order လိုက် middleware များ ချိတ်ဆက်ဆောင်ရွက်မှု
- **Context Propagation**: middleware အဆုံးဆက်တွင် အခြေအနေကို ဖြတ်သန်းပို့ဆောင်ခြင်း


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
